In [1]:
# Instalación de librerías esenciales para modelos tabulares en producción
!pip install -q transformers datasets accelerate scikit-learn pandas numpy huggingface_hub --quiet

import torch
# Configuración dinámica de hardware: USA GPU si está activa para el procesamiento de lotes
device = 0 if torch.cuda.is_available() else -1
print(f"[INFO] Hardware listo. Trabajando en el dispositivo: {'GPU' if device == 0 else 'CPU'}")


[INFO] Hardware listo. Trabajando en el dispositivo: GPU


In [2]:
import time
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from huggingface_hub import notebook_login

# Autenticación requerida como en la clase
notebook_login()


In [54]:

import numpy as pd
import pandas as pd

print("Cargando datos...")

# URL Archivo CSV
df_real = pd.read_csv("/content/Actividad_4_Gestion_Proyectos.csv")
df_real.columns = df_real.columns.str.strip()


if "Fecha" in df_real.columns:
    df_real = df_real.drop(columns=["Fecha"])

# Limpiar errores
for col in df_real.columns:
    df_real[col] = pd.to_numeric(df_real[col], errors='coerce')
df_real = df_real.dropna()

# Aqui ponemos las constraints para el horno
df_real["eficiencia_co2"] = np.where(
    (df_real["%CO"] < 0.05) &
    (df_real["%Gases NO"] < 600) &
    (df_real["%O2"] < 4.2) &
    (df_real["Potencia Motor"] <= 280.0),
    1, 0
)

#Solo va a manipular las variables de procesos, va a manipular 'Alimentacion', 'VTI Precalentador', 'VTI Enfriador', 'Flujo de Carbon quemador', 'Flujo Carbon Calcinador', 'Potencia Motor', 'Llenado de Horno'
columnas_restricciones_y_salidas = ["%Gases NO", "%Gases SO2", "%CO", "%O2", "Potencia Motor", "eficiencia_co2"]
X = df_real.drop(columns=columnas_restricciones_y_salidas)
y = df_real["eficiencia_co2"]

print(f"\n[OK] Variables manipuladas para entrenamiento (X):")
print(list(X.columns))
print(f"\nTotal registros del SCADA: {len(df_real)}")
print(f"--> Estados de Combustión Óptima y Limpia (1): {sum(y == 1)}")
print(f"--> Estados Ineficientes / Exceso Emisiones (0): {sum(y == 0)}")


Cargando datos...

[OK] Variables manipuladas para entrenamiento (X):
['Alimentacion', 'VTI Precalentador', 'VTI Enfriador', 'Flujo de Carbon quemador', 'Flujo Carbon Calcinador', 'Llenado de Horno']

Total registros del SCADA: 9184
--> Estados de Combustión Óptima y Limpia (1): 390
--> Estados Ineficientes / Exceso Emisiones (0): 8794


In [58]:

import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

#Usamos los pesos del Finetuning basados en el desbalance de los sensores ya que buscamos condiciones no tan comunes en el horno
# Esto le dice al modelo que los casos eficientes  valen más matemáticamente
pesos_fine_tuning = compute_sample_weight(class_weight='balanced', y=y)

# Ajustes de hiperparametros
modelos_horno = {
    "HF-Sklearn/Gradient-Boosting-Furnace-Core-FinedTuned": GradientBoostingClassifier(
        n_estimators=300,       # Aumentamos los árboles de decision
        learning_rate=0.05,     # Ajuste fino más lento y preciso
        max_depth=5,            # Mayor tolerancia a interacciones complejas de gases
        random_state=42
    ),
    "HF-Sklearn/Random-Forest-Emission-Monitor-Balanced": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced", # Fine tunning automático de pesos por árbol
        random_state=42
    ),
    "HF-Sklearn/Logistic-Regression-Industrial-Balanced-Baseline": LogisticRegression(
        max_iter=10000,
        class_weight="balanced", # Forzar linea rect a buscar valores raros
        random_state=42
    )
}

def evaluar_control_horno_finetuned(model_name, model_obj, X_data, y_true, sample_weights):
    # Hacemos fintuning inyectando los pesos balanceados del SCADA
    if "Gradient-Boosting" in model_name:
        model_obj.fit(X_data, y_true, sample_weight=sample_weights)
    else:
        model_obj.fit(X_data, y_true) # Los otros modelos ya llevan el balanced interno

    # Latencia inferencia
    start_time = time.time()
    predictions = model_obj.predict(X_data)
    end_time = time.time()

    total_latency = end_time - start_time
    avg_latency = total_latency / len(X_data)

    # CCalculo de la metricas
    accuracy = accuracy_score(y_true, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, predictions, average='binary', zero_division=0)

    return {
        "Modelo Tabular (HF Fine-Tuned)": model_name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "Latencia Total (s)": round(total_latency, 4),
        "Latencia Promedio (s/m)": round(avg_latency, 6)
    }


In [56]:
resultados_proyecto = []

print("=== EVALUANDO OPERACIÓN EN TIEMPO REAL CON VARIABLES DEL HORNO ===")

# Ejecutamos el bucle secuencial sobre los tres modelos
for name, model_obj in modelos_horno.items():
    print(f"Verificando variables manipuladas frente a restricciones con: {name}...")
    res = evaluar_control_horno(name, model_obj, X, y)
    resultados_proyecto.append(res)

# Consolidación de los resultados en un DataFrame de Pandas
df_resumen_final = pd.DataFrame(resultados_proyecto)

print("\n" + "="*95)
print("TABLA COMPARATIVA DE MODELOS - PROYECTO OPTIMIZACIÓN DE CO2 EN HORNOS")
print("="*95)
print(df_resumen_final.to_string(index=False))
print("="*95)


=== EVALUANDO OPERACIÓN EN TIEMPO REAL CON VARIABLES DEL HORNO ===
Verificando variables manipuladas frente a restricciones con: HF-Sklearn/Gradient-Boosting-Furnace-Core-FinedTuned...
Verificando variables manipuladas frente a restricciones con: HF-Sklearn/Random-Forest-Emission-Monitor-Balanced...
Verificando variables manipuladas frente a restricciones con: HF-Sklearn/Logistic-Regression-Industrial-Balanced-Baseline...

TABLA COMPARATIVA DE MODELOS - PROYECTO OPTIMIZACIÓN DE CO2 EN HORNOS
                                        Modelo Tabular (HF)  Accuracy  Precision  Recall  F1-Score  Latencia Total (s)  Latencia Promedio (s/m)
       HF-Sklearn/Gradient-Boosting-Furnace-Core-FinedTuned    0.9763     0.9831  0.4487    0.6162              0.0514                 0.000006
         HF-Sklearn/Random-Forest-Emission-Monitor-Balanced    1.0000     1.0000  1.0000    1.0000              0.1692                 0.000018
HF-Sklearn/Logistic-Regression-Industrial-Balanced-Baseline    0.6498  

In [57]:

print("=== Recomendaciones ===")

# Filtrar los datos que son reales
df_eficiente = df_real[df_real["eficiencia_co2"] == 1]

if not df_eficiente.empty:
    # Variables manipulables
    variables_manipulables = list(X.columns)

    print(f"\n Para mantener el %CO < 0.05, %Gases NO < 500 y %O2 < 3,")
    print("el sistema SCADA y los operadores deben mantener las variables en los siguientes rangos:\n")

    recomendaciones = []
    for var in variables_manipulables:
        val_min = df_eficiente[var].min()
        val_max = df_eficiente[var].max()
        val_promedio = df_eficiente[var].mean()

        recomendaciones.append({
            "Variable Manipulable": var,
            "Valor Mínimo": round(val_min, 4),
            "Valor Máximo": round(val_max, 4),
            "Setpoint Sugerido (Promedio)": round(val_promedio, 4)
        })

    # Presentamos las respuestas en una tabla limpia de recomendaciones operativas
    df_setpoints = pd.DataFrame(recomendaciones)
    print(df_setpoints.to_string(index=False))
    print("\n" + "="*70)
    print("Estos valores garantizan una mejor eficiencia minimizando la huella de CO2.")
else:
    print("Ningún registro cumplió con las 3 restricciones ambientales estrictas simultáneamente.")

=== Recomendaciones ===

 Para mantener el %CO < 0.05, %Gases NO < 500 y %O2 < 3,
el sistema SCADA y los operadores deben mantener las variables en los siguientes rangos:

    Variable Manipulable  Valor Mínimo  Valor Máximo  Setpoint Sugerido (Promedio)
            Alimentacion        0.0000      190.7145                      142.1151
       VTI Precalentador        0.0000      677.3420                      470.6312
           VTI Enfriador        0.0056      407.2898                      203.7902
Flujo de Carbon quemador        0.3000        6.1000                        3.9610
 Flujo Carbon Calcinador        0.0000        9.0238                        6.4971
        Llenado de Horno        0.0000       24.5000                       12.8577

Estos valores garantizan una mejor eficiencia minimizando la huella de CO2.
